# agri-drone — Notebook 3: fair multi-backbone baseline re-audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashut0sh-mishra/agri-drone/blob/main/notebooks/colab/03_baseline_reaudit.ipynb)

Trains EfficientNet-B0, ConvNeXt-Tiny, MobileNetV3-Small under the shared recipe in `docs/training_recipe.md@v1`.
Weights save to `MyDrive/agri-drone/models_v2/` (NEVER committed).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!test -d agri-drone || git clone https://github.com/Ashut0sh-mishra/agri-drone.git
%cd /content/agri-drone
!git fetch --all && git checkout main && git pull
!pip install -q -r requirements.txt timm


In [ ]:
import torch
assert torch.cuda.is_available(), 'Need GPU runtime'
DRIVE = '/content/drive/MyDrive/agri-drone'
DATA  = f'{DRIVE}/data/plantvillage'
OUT_MODELS = f'{DRIVE}/models_v2'
OUT_RESULTS = f'{DRIVE}/results_v2/baselines'
import os; os.makedirs(OUT_MODELS, exist_ok=True); os.makedirs(OUT_RESULTS, exist_ok=True)
print('data:', DATA)


## Shared training recipe (`docs/training_recipe.md@v1`)


In [ ]:
RECIPE = dict(
    optimizer='AdamW', lr=1.25e-3, weight_decay=0.05,
    scheduler='cosine', warmup_epochs=3, epochs=50,
    batch_size=32, label_smoothing=0.1,
    augmentation='standard_v1', precision='fp16', seed=42,
)
print(RECIPE)


## Train EfficientNet-B0


In [ ]:
!python evaluate/matrix/audit_baseline.py --backbone efficientnet_b0 --out-dir $OUT_RESULTS


## Train ConvNeXt-Tiny


In [ ]:
!python evaluate/matrix/audit_baseline.py --backbone convnext_tiny --out-dir $OUT_RESULTS


## Train MobileNetV3-Small


In [ ]:
!python evaluate/matrix/audit_baseline.py --backbone mobilenetv3_small --out-dir $OUT_RESULTS


## Assemble BASELINES_TABLE.md


In [ ]:
import json, pathlib, glob
rows = []
for f in sorted(glob.glob(f'{OUT_RESULTS}/**/audit.json', recursive=True)):
    d = json.loads(open(f).read())
    rows.append((d.get('backbone','?'), d.get('accuracy','[TBD]'), d.get('macro_f1','[TBD]'), d.get('params_m','[TBD]'), d.get('latency_ms','[TBD]')))
md = ['# \u00a76.7 Fair multi-backbone baselines', '',
      '| Backbone | Accuracy | Macro-F1 | Params (M) | Latency (ms) |',
      '|---|---:|---:|---:|---:|']
for r in rows:
    md.append('| ' + ' | '.join(str(x) for x in r) + ' |')
md.append('')
md.append('All trained under docs/training_recipe.md@v1.')
pathlib.Path(f'{OUT_RESULTS}/BASELINES_TABLE.md').write_text('\n'.join(md))
print('\n'.join(md))
